# Anki 카드 보강 모델 미세조정 (Qwen3-8B + LoRA)

`scripts/export_sft_dataset.py`가 만든 데이터로, 앱의 `complete_fields` 요청(뜻·품사·예문·설명 / 해설 / 문법 보기)을 처리하는 로컬 모델을 학습한다.

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**. 데이터는 `MyDrive/anki-sft/data/{train,val,test}.jsonl`.

순서: 학습 → LoRA 어댑터 저장 → test 60건 자동 채점 → GGUF(q4_k_m) 변환 → Drive 저장 → 런타임 자동 종료.
결과는 전부 `MyDrive/anki-sft/runs/<시각>/`에 남는다 (구독이 끝나도 파일은 내 것).

예상: 학습 토큰 약 1,010만 개(1 epoch), 1시간 안팎 + GGUF 변환 15~30분 (미검증 추정). A100 80GB 기준 시간당 약 7 CU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q unsloth
import importlib.metadata as md
for p in ["unsloth", "trl", "transformers", "peft", "torch"]:
    print(p, md.version(p))  # 다시 돌릴 때 같은 버전을 쓰도록 기록해 둔다

In [ ]:
import json, time, datetime, os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/anki-sft/data"
RUN = f"/content/drive/MyDrive/anki-sft/runs/{datetime.datetime.now():%Y%m%d-%H%M}"
os.makedirs(RUN, exist_ok=True)

MODEL_NAME = "unsloth/Qwen3-8B"   # Apache 2.0
MAX_LEN = 1024                     # 데이터 최대 964토큰 (2026-11-22 측정)
EPOCHS = 1
VAL_LIMIT = 300                    # eval loss 계산용 (전체 val을 쓰면 느리다)
TEST_EVAL_N = 60
SEED = 3407

def read(split):
    with open(f"{DATA}/{split}.jsonl", encoding="utf-8") as fh:
        return [json.loads(l) for l in fh]

train_rows, val_rows, test_rows = read("train"), read("val"), read("test")
print({k: len(v) for k, v in [("train", train_rows), ("val", val_rows), ("test", test_rows)]})
t0 = time.time()

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_LEN, dtype=None, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)

In [ ]:
from datasets import Dataset

def to_text(row):
    # Qwen3의 생각(thinking) 모드는 끈다: 카드 보강은 바로 JSON을 내야 한다
    return tokenizer.apply_chat_template(row["messages"], tokenize=False, enable_thinking=False)

def build(rows):
    texts = [to_text(r) for r in rows]
    lens = [len(tokenizer(t).input_ids) for t in texts]
    kept = [t for t, n in zip(texts, lens) if n <= MAX_LEN]
    print(f"{len(kept)}/{len(texts)} kept, max tokens {max(lens)}, mean {sum(lens)/len(lens):.0f}")
    return Dataset.from_list([{"text": t} for t in kept])

train_ds = build(train_rows)
val_ds = build(val_rows[:VAL_LIMIT])
print(train_ds[0]["text"][-600:])

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# trl 버전마다 인자 이름이 달라서 있는 쪽을 쓴다
cfg_params = inspect.signature(SFTConfig.__init__).parameters
cfg = dict(
    dataset_text_field="text", per_device_train_batch_size=8, gradient_accumulation_steps=2,
    num_train_epochs=EPOCHS, learning_rate=2e-4, lr_scheduler_type="linear", warmup_ratio=0.03,
    logging_steps=20, eval_steps=200, save_steps=200, save_total_limit=2,
    output_dir="/content/checkpoints", bf16=True, report_to="none", seed=SEED)
cfg["max_length" if "max_length" in cfg_params else "max_seq_length"] = MAX_LEN
cfg["eval_strategy" if "eval_strategy" in cfg_params else "evaluation_strategy"] = "steps"

tr_params = inspect.signature(SFTTrainer.__init__).parameters
tok_key = "processing_class" if "processing_class" in tr_params else "tokenizer"
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds,
                     args=SFTConfig(**cfg), **{tok_key: tokenizer})
# 질문(시스템 프롬프트·요청문)은 손실에서 빼고 답(JSON)만 배운다
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

In [ ]:
stats = trainer.train()
print(stats)
model.save_pretrained(f"{RUN}/lora")
tokenizer.save_pretrained(f"{RUN}/lora")
json.dump(trainer.state.log_history, open(f"{RUN}/log_history.json", "w"), ensure_ascii=False, indent=1)
print("train minutes:", round((time.time() - t0) / 60, 1))

## test 자동 채점

JSON으로 파싱되는지, 요청한 필드(missing)를 정확히 돌려주는지, 단어 카드 뜻에 슬래시가 없는지를 센다. 내용의 정확도는 여기서 판단하지 않는다 (나중에 Flash-Lite와 나란히 비교).

In [ ]:
import random
FastLanguageModel.for_inference(model)
rnd = random.Random(SEED)
sample = rnd.sample(test_rows, min(TEST_EVAL_N, len(test_rows)))
results, n_json, n_keys, n_slash = [], 0, 0, 0
for row in sample:
    prompt = tokenizer.apply_chat_template(row["messages"][:2], tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=700, do_sample=False)
    text = tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()
    try:
        ans = json.loads(text); n_json += 1
        n_keys += set(ans) == set(row["meta"]["missing"])
        n_slash += "/" in str(ans.get("뜻", ""))
    except Exception:
        ans = None
    results.append({"note_id": row["meta"]["note_id"], "model": row["meta"]["model"],
                    "expected": row["messages"][2]["content"], "got": text})
n = len(sample)
summary = {"n": n, "json_ok": n_json, "keys_exact": n_keys, "slash_in_meaning": n_slash}
print(summary)
json.dump({"summary": summary, "results": results}, open(f"{RUN}/test_eval.json", "w"), ensure_ascii=False, indent=1)
for r in results[:3]:
    print("---", r["model"]); print("expected:", r["expected"][:300]); print("got     :", r["got"][:300])

In [ ]:
# 맥(16GB)에서 Ollama/llama.cpp로 돌릴 4비트 GGUF
model.save_pretrained_gguf("/content/gguf", tokenizer, quantization_method="q4_k_m")
for path in glob.glob("/content/gguf*/**/*.gguf", recursive=True) + glob.glob("/content/*.gguf"):
    print("copy", path, round(os.path.getsize(path) / 1e9, 2), "GB")
    shutil.copy(path, RUN)
print(os.listdir(RUN))
print("total minutes:", round((time.time() - t0) / 60, 1))

In [ ]:
# 결과가 Drive에 있는지 확인한 뒤 실행: 런타임을 끊어 CU가 더 빠지지 않게 한다
from google.colab import runtime
runtime.unassign()